# TP3. Polynômes et fonctions

Carnet de travail. L'énoncé complet, avec les corrigés, est sur le site :
https://cours.gclab.fr/outils-fondamentaux/tp3-polynomes-fonctions/

Exécutez une cellule avec Maj+Entrée. **Pensez à télécharger ce carnet avant de partir** : menu File, puis Download.


## Exercice 1.1 : évaluation naïve

Écrivez `evaluer_naif(P, x)` qui calcule $P(x)$ en suivant la définition, avec une
puissance par terme.

**Résultat attendu :** `evaluer_naif([2, 3, 0, 5], 2)` vaut `48`, car
$2 + 6 + 40 = 48$.


In [ ]:
def evaluer_naif(P, x):
    result = 0
    for i, a in enumerate(P):
        result += a * x**i
    return result

print(evaluer_naif([2, 3, 0, 5], 2))


## Exercice 2.1 : Horner

Écrivez `evaluer(P, x)` par le schéma de Horner, et vérifiez qu'elle donne le même
résultat que la version naïve sur plusieurs points.

**Résultat attendu :** les deux fonctions coïncident, et Horner est plus rapide sur un
polynôme de degré élevé.


In [ ]:
def evaluer(P, x):
    result = 0
    for a in reversed(P):
        result = result * x + a
    return result

print(evaluer([2, 3, 0, 5], 2))


## Exercice 2.2 : mesurer, et se méfier de ses prévisions

Construisez des polynômes dont tous les coefficients valent 1, de degrés 1 000, 5 000
et 20 000, et comparez le temps des deux méthodes en `x = 1.0001`. Avant de lancer la
mesure, écrivez votre prévision : de combien Horner devrait-il être plus rapide ?

**Résultat attendu :** un rapport d'environ 2, **constant** quand le degré augmente.
Ce n'est probablement pas ce que vous aviez prévu.


In [ ]:
import time

for deg in [1000, 5000, 20000]:
    P = [1] * (deg + 1)
    t0 = time.time()
    evaluer_naif(P, 1.0001)
    t1 = time.time()
    t2 = time.time()
    evaluer(P, 1.0001)
    t3 = time.time()
    naif = t1 - t0
    horn = t3 - t2
    print(f"deg={deg}: naif={naif*1000:.1f}ms horner={horn*1000:.1f}ms rapport={naif/horn:.1f}")


## Exercice 2.3 : le cas où l'écart explose vraiment

Refaites la mesure en évaluant en `x = 3`, un **entier**, sur des degrés 200, 400
et 800. Le rapport reste-t-il constant ?

**Résultat attendu :** cette fois le rapport grandit avec le degré, d'environ 4 à
environ 6.


In [ ]:
import time

for deg in [200, 400, 800]:
    P = [1] * (deg + 1)
    t0 = time.time()
    evaluer_naif(P, 3)
    t1 = time.time()
    t2 = time.time()
    evaluer(P, 3)
    t3 = time.time()
    naif = t1 - t0
    horn = t3 - t2
    print(f"deg={deg} x=3: rapport={naif/horn:.1f}")


## Exercice 3.1 : la dérivée

Écrivez `deriver(P)` qui renvoie la liste des coefficients du polynôme dérivé.

**Résultat attendu :** `deriver([2, 3, 0, 5])` vaut `[3, 0, 15]`, c'est-à-dire
$3 + 15x^{2}$.


In [ ]:
def deriver(P):
    return [(i + 1) * P[i + 1] for i in range(len(P) - 1)]

print(deriver([2, 3, 0, 5]))


## Exercice 4.1 : dichotomie

Écrivez `dichotomie(f, a, b, tolerance)` qui renvoie une racine de `f` entre `a` et
`b`. Appliquez-la à $x^{2} - 2$ sur $[0, 2]$.

**Résultat attendu :** une valeur proche de $\sqrt{2} \approx 1{,}414214$.


In [ ]:
def dichotomie(f, a, b, tolerance=1e-10):
    while b - a > tolerance:
        m = (a + b) / 2
        if f(a) * f(m) <= 0:
            b = m
        else:
            a = m
    return (a + b) / 2

print(dichotomie(lambda x: x**2 - 2, 0, 2))


## Exercice 4.2 : Newton sur un polynôme

Écrivez `newton(P, depart, tolerance)` qui cherche une racine du polynôme `P` en
utilisant `evaluer` et `deriver`. Appliquez-la à $x^{3} - 2x - 5$, en partant de 2.

**Résultat attendu :** environ `2.0945515`, atteint en cinq tours ou moins.


In [ ]:
def newton(P, depart, tolerance=1e-10, max_iter=100):
    x = depart
    dP = deriver(P)
    for _ in range(max_iter):
        fx = evaluer(P, x)
        dfx = evaluer(dP, x)
        if dfx == 0:
            break
        x_new = x - fx / dfx
        if abs(x_new - x) < tolerance:
            return x_new
        x = x_new
    return x

print(newton([-5, -2, 0, 1], 2))


## Exercice 4.3 : faire échouer Newton

Appliquez `newton` au polynôme $x^{3} - 2x + 2$, c'est-à-dire `[2, -2, 0, 1]`, en
partant de `0`. Affichez le trajet des premiers itérés pour comprendre ce qui se passe.

**Résultat attendu :** aucune convergence, et un trajet qui alterne entre `0` et `1`
indéfiniment.


In [ ]:
x = 0.0
dP = deriver([2, -2, 0, 1])
for i in range(10):
    fx = evaluer([2, -2, 0, 1], x)
    dfx = evaluer(dP, x)
    print(f"iter {i}: x={x}")
    if dfx == 0:
        break
    x = x - fx / dfx


## Exercice 4.4 : le prix d'un mauvais point de départ

Reprenez $x^{3} - 2x - 5$ et comparez le nombre de tours de `newton` en partant
de `2`, puis de `0`.

**Résultat attendu :** cinq tours depuis `2`, une vingtaine depuis `0`.


In [ ]:
def newton_compte(P, depart, tolerance=1e-10, max_iter=200):
    x = depart
    dP = deriver(P)
    for i in range(max_iter):
        fx = evaluer(P, x)
        dfx = evaluer(dP, x)
        if dfx == 0:
            return x, i
        x_new = x - fx / dfx
        if abs(x_new - x) < tolerance:
            return x_new, i + 1
        x = x_new
    return x, max_iter

r1, n1 = newton_compte([-5, -2, 0, 1], 2)
r2, n2 = newton_compte([-5, -2, 0, 1], 0)
print(f"depuis 2: {n1} tours")
print(f"depuis 0: {n2} tours")


## Exercice 5.1 : lire le graphique

Sur le tracé, repérez la racine trouvée par Newton. Combien de fois la courbe
coupe-t-elle l'axe des abscisses ? Ce polynôme est de degré 3 : que dire de ses autres
racines ?


In [ ]:
print("La courbe coupe l'axe une seule fois : une racine reelle, les deux autres sont complexes.")


## Exercice 6.1 : somme et produit

Écrivez `somme(P, Q)` et `produit_poly(P, Q)`. Attention, les deux listes n'ont pas
forcément la même longueur.

**Résultat attendu :** `produit_poly([1, 1], [1, 1])` vaut `[1, 2, 1]`, c'est-à-dire
$(1 + x)^{2} = 1 + 2x + x^{2}$, et `produit_poly([-1, 1], [1, 1])` vaut
`[-1, 0, 1]`, l'identité $(x - 1)(x + 1) = x^{2} - 1$.


In [ ]:
def somme(P, Q):
    n = max(len(P), len(Q))
    result = []
    for i in range(n):
        a = P[i] if i < len(P) else 0
        b = Q[i] if i < len(Q) else 0
        result.append(a + b)
    return result

def produit_poly(P, Q):
    result = [0] * (len(P) + len(Q) - 1)
    for i, a in enumerate(P):
        for j, b in enumerate(Q):
            result[i + j] += a * b
    return result

print(produit_poly([1, 1], [1, 1]))
print(produit_poly([-1, 1], [1, 1]))


## Exercice 6.2 : le triangle de Pascal, par les polynômes

Calculez $(1 + x)^{10}$ en multipliant dix fois `[1, 1]` par lui-même. Que
reconnaissez-vous dans la liste obtenue ?

**Résultat attendu :** `[1, 10, 45, 120, 210, 252, 210, 120, 45, 10, 1]`.


In [ ]:
p = [1, 1]
for _ in range(9):
    p = produit_poly(p, [1, 1])
print(p)


## Exercice 7.1 : trouver le meilleur pas

Écrivez `derivee_avant(f, x, h)` et `derivee_centree(f, x, h)`. Calculez l'erreur de
chacune sur $\sin$ en $x = 1$, dont la dérivée exacte est $\cos 1$, pour
`h` valant $10^{-1}, 10^{-2}, 10^{-4}, 10^{-6}, 10^{-8}, 10^{-10}, 10^{-12}$.

**Résultat attendu :** l'erreur diminue puis **remonte**. Le meilleur pas est de
l'ordre de $10^{-8}$ pour la formule avant, $10^{-6}$ pour la formule centrée,
et à $h = 10^{-12}$ l'erreur est remontée de quatre à six ordres de grandeur
par rapport à ce meilleur pas.


In [ ]:
import math

def derivee_avant(f, x, h):
    return (f(x + h) - f(x)) / h

def derivee_centree(f, x, h):
    return (f(x + h) - f(x - h)) / (2 * h)

x = 1.0
exacte = math.cos(1)
for h in [1e-1, 1e-2, 1e-4, 1e-6, 1e-8, 1e-10, 1e-12]:
    ea = abs(derivee_avant(math.sin, x, h) - exacte)
    ec = abs(derivee_centree(math.sin, x, h) - exacte)
    print(f"h={h:.0e}: avant={ea:.2e} centree={ec:.2e}")


## Exercice 8.1 : les trois méthodes

Écrivez `rectangles(f, a, b, n)`, `trapezes(f, a, b, n)` et `simpson(f, a, b, n)`.
Testez-les sur $\int_0^1 x^{2}\,dx = 1/3$ pour `n` valant 10, 100 et 1000.

**Résultat attendu :** l'erreur des rectangles est divisée par 10 quand `n` est
multiplié par 10, celle des trapèzes par 100, et Simpson donne le résultat exact, à
un arrondi de $10^{-17}$ près.


In [ ]:
def rectangles(f, a, b, n):
    h = (b - a) / n
    return sum(f(a + i * h) for i in range(n)) * h

def trapezes(f, a, b, n):
    h = (b - a) / n
    return (f(a) / 2 + sum(f(a + i * h) for i in range(1, n)) + f(b) / 2) * h

def simpson(f, a, b, n):
    h = (b - a) / n
    total = f(a) + f(b)
    for i in range(1, n):
        total += (4 if i % 2 == 1 else 2) * f(a + i * h)
    return total * h / 3

for n in [10, 100, 1000]:
    r = rectangles(lambda x: x**2, 0, 1, n)
    t = trapezes(lambda x: x**2, 0, 1, n)
    s = simpson(lambda x: x**2, 0, 1, n)
    print(f"n={n}: rect={abs(r-1/3):.2e} trap={abs(t-1/3):.2e} simpson={abs(s-1/3):.2e}")


## Exercice 8.2 : calculer π

On a $\int_0^1 \frac{4}{1 + x^{2}}\,dx = \pi$. Calculez cette intégrale par les
trapèzes et par Simpson avec `n = 1000`, et comparez à `math.pi`.

**Résultat attendu :** une erreur de l'ordre de $10^{-7}$ pour les trapèzes, et
un résultat indistinguable de `math.pi` pour Simpson.


In [ ]:
import math

f_pi = lambda x: 4 / (1 + x**2)
print(f"trapezes: erreur={abs(trapezes(f_pi, 0, 1, 1000) - math.pi):.2e}")
print(f"simpson:  erreur={abs(simpson(f_pi, 0, 1, 1000) - math.pi):.2e}")
